# Step 2: Cohort Selection, Feature Engineering, and Target Labeling

## Overview
This notebook constructs the study cohort of patients with solid tumors receiving chemotherapy, calculates baseline clinical features prior to chemotherapy administration (Day 1), and labels the target composite outcome (chemotherapy-induced neutropenic event) within a 28-day window.

### 1. Cohort Selection Criteria
- **Inclusion**: Patients with solid tumor diagnoses.
- **Exclusion**: Patients diagnosed with hematological malignancies (ICD-O-3 codes: `C42`, `C81-C89`, `C90-C96`) at baseline are excluded to focus the analysis on solid tumors.

### 2. Composite Outcome Definition (Target Label $Y \in \{0, 1\}$)
A positive outcome ($Y=1$) is defined as the occurrence of any of the following within 28 days post-chemotherapy (Day 1 to Day 28):
- **Severe Neutropenia**: Absolute Neutrophil Count (ANC) $< 0.5 	imes 10^9/\text{L}$ (Grade 4).
- **Febrile Neutropenia (FN)**: ANC $< 1.0 	imes 10^9/\text{L}$ accompanied by a fever ($BT \ge 38.5^\circ\text{C}$).
- **Therapeutic G-CSF**: Administration of G-CSF agents (G-CSF codes: `3399405`, `3399406`, `3399408`) indicated for neutropenia management.

### 3. Features Generated ($X$)
- **Laboratory Values**: Baseline ANC (`lastNEUT#`) and WBC (`lastWBC`).
- **Somatometry**: Latest height and weight.
- **Vital Signs**: Body temperature (`TEMPR`), pulse rate (`PULSE`), respiration rate (`RESP`), systolic/diastolic blood pressure (`BPH`/`BPL`).
- **Chemotherapy Exposure**: Cumulative dosages of individual chemotherapeutic drugs.
- **Prophylactic G-CSF**: Prior dosage of G-CSF (`3399410`) within the window of $[ \text{Day } 1 - 3, \text{Day } 1 )$.
- **Demographics**: Patient age and sex.

## Inputs and Outputs
- **Input**: Processed step1 files: `../../data/step1_dwh_outputs/step1_all_*.parquet`
- **Output**: `../../data/step2_processing_outputs/step2_all_tagt_feat`



In [ ]:
import pandas as pd
import numpy as np
import os
import re

In [ ]:
# ==========================================
# 1. Setting and Constant Definitions
# ==========================================
DATA_DIR = '../../data'
INPUT_DIR = f'{DATA_DIR}/step1_dwh_outputs'
OUTPUT_DIR = f'{DATA_DIR}/step2_processing_outputs'

# File mapping definitions of aggregated Parquet files
FILE_NAMES = {
    '検査値': 'step1_all_sample_rslt.parquet',
    '注射実施': 'step1_all_injection.parquet',
    '内服実施': 'step1_all_drug_taken.parquet',
    'バイタル': 'step1_all_vital.parquet',
    '身体計測': 'step1_all_somatometry.parquet',
    'がん登録': 'step1_all_cancer.parquet',
}

# Drug classification codes
GCSF_CODES = ["3399405", "3399406", "3399408"]  # Drugs used for therapeutic G-CSF administration
GLAS_CODES = ["3399410"]                        # Drugs used for prophylactic G-CSF administration
GLAS_CODES = ["3399410"]                        
REQ_EXAM_ITEMS = {'  NEUT#', '  WBC'}
CANCER_EXCLUDE_PATTERN = r'^C(42|8[1-9]|9[0-6])$'

# Clinical thresholds
TH_NEUT_SEVERE = 0.5
TH_NEUT_MODERATE = 1.0
TH_FEVER = 38.5

# Time window parameters (Day 1 definition)
WINDOW_EVENT_END_DAY = 28 
WINDOW_PRE_GCSF_DAYS = 3  

In [ ]:
# ==========================================
# 2. General Helper Functions 
# ==========================================
def load_integrated_data(input_dir, file_mapping):
    """
    統合済みParquetファイルを直接読み込む関数
    """
    dfs = {}
    print(f"データ読み込み開始: {input_dir}")

    for key, file_name in file_mapping.items():
        file_path = os.path.join(input_dir, file_name)
        
        if os.path.exists(file_path):
            try:
                # 読み込み
                df = pd.read_parquet(file_path)
                dfs[key] = df
                print(f"読み込み完了: {key} (Records: {len(df)})")
            except Exception as e:
                print(f"Error: {file_path} の読み込み失敗: {e}")
                dfs[key] = pd.DataFrame()
        else:
            # 拡張子 .parquet を付与して再確認
            file_path_with_ext = file_path + '.parquet'
            if os.path.exists(file_path_with_ext):
                try:
                    df = pd.read_parquet(file_path_with_ext)
                    dfs[key] = df
                    print(f"読み込み完了(.parquet付与): {key} (Records: {len(df)})")
                except Exception as e:
                    print(f"Error: {file_path_with_ext} の読み込み失敗: {e}")
                    dfs[key] = pd.DataFrame()
            else:
                print(f"Warning: ファイルが見つかりません: {file_path}")
                dfs[key] = pd.DataFrame()

    return dfs

def get_latest_record_before_target(df_target_dates, df_source, date_col_source, target_date_col='DO_DATE', on='hs'):
    """Retrieve the latest record on or prior to the reference date (DO_DATE)"""
    merged = pd.merge(df_target_dates[[on, target_date_col]].drop_duplicates(), 
                      df_source, on=on)
    
    filtered = merged[merged[date_col_source] <= merged[target_date_col]].copy()
    filtered['MAX_DATE'] = filtered.groupby([on, target_date_col])[date_col_source].transform('max')
    
    return filtered[filtered[date_col_source] == filtered['MAX_DATE']].drop(columns=['MAX_DATE'])

In [ ]:
# ==========================================
# 3. Data Processing Logic
# ==========================================
def process_drugs(dfs):
    """Preprocess and clean drug administration records"""
    def _clean_drug(df):
        df = df.copy()
        # Drop records with blank DRUG_PRICE_LIST_CD
        df = df[df["DRUG_PRICE_LIST_CD"].str.strip() != ""]
        # Extract the first 7 digits of the drug code 
        df['DRUG_CD'] = df['DRUG_PRICE_LIST_CD'].str.slice(0, 7)
        return df.rename(columns={'QUERY_DATE': 'DO_DATE'}) 
       

    df_inje = _clean_drug(dfs['注射実施'])
    df_drug = _clean_drug(dfs['内服実施'])

    # Therapeutic G-CSF (used for event/label determination)
    df_gcsf_therapeutic = df_inje.loc[df_inje['DRUG_CD'].isin(GCSF_CODES), ['hs', 'DO_DATE', 'DRUG_CD', 'EXEC_DOSE']].copy()
    df_gcsf_therapeutic = df_gcsf_therapeutic.rename(columns={'DO_DATE': 'GCSF_DATE'})

    # Prophylactic G-CSF (used for feature generation)
    df_glas_prophylactic = df_inje.loc[df_inje['DRUG_CD'].isin(GLAS_CODES), ['hs', 'DO_DATE', 'DRUG_CD', 'EXEC_DOSE']].copy()
    df_glas_prophylactic = df_glas_prophylactic.rename(columns={'DO_DATE': 'GLAS_DATE'})

    # Chemotherapy administration data
    df_chemo = pd.concat([df_drug, df_inje])
    df_chemo = df_chemo[df_chemo["ANTI_CANCER_FLG"] == "＋"].copy()
    
    return df_chemo, df_gcsf_therapeutic, df_glas_prophylactic

def filter_cancer_patients(df_canc):
    """Filter out patients with diagnosis codes targeting exclusion (e.g., hematological cancers)"""
    df = df_canc.copy()
    df['DIAGNOSIS_DATE_2'] = pd.to_datetime(df['DIAGNOSIS_DATE_2'], errors='coerce')
    df['DIAGNOSIS_CD_SHORT'] = df['DIAGNOSIS_CD'].str.slice(0, 3)
    
    min_dates = df.groupby('hs')['DIAGNOSIS_DATE_2'].transform('min')
    earliest_records = df[df['DIAGNOSIS_DATE_2'] == min_dates]
    
    exclude_mask = earliest_records['DIAGNOSIS_CD_SHORT'].str.match(CANCER_EXCLUDE_PATTERN, na=False)
    hs_to_drop = earliest_records.loc[exclude_mask, 'hs'].unique()
    
    df_filtered = df[~df['hs'].isin(hs_to_drop)].copy()
    df_dummies = pd.get_dummies(df_filtered[['hs', 'DIAGNOSIS_CD_SHORT']], 
                                columns=['DIAGNOSIS_CD_SHORT'], prefix='DIAGNOSIS')
    return df_dummies.groupby('hs').max().reset_index()

def create_event_labels(df_chemo, df_exam, df_vt, df_gcsf_therapeutic):
    """Generate target outcome labels (Composite Event)"""
    # Aggregate laboratory values and vital signs
    df_exam_min = (df_exam[df_exam['EXAM_ITEM'].isin(REQ_EXAM_ITEMS)]
                   .groupby(['hs', 'QUERY_DATE', 'EXAM_ITEM'])['RESULT_NUM'].min()
                   .unstack().reset_index()
                   .rename(columns={'  NEUT#': 'minNEUT#', '  WBC': 'minWBC'}))
    
    df_vt_max = (df_vt.groupby(['hs', 'QUERY_DATE'])['TEMPR'].max()
                 .reset_index(name='maxBT'))
    
    df_daily = pd.merge(df_exam_min, df_vt_max, on=['hs', 'QUERY_DATE'], how='outer')

    # Determine Febrile Neutropenia (FN) event occurrence
    condition_severe = df_daily['minNEUT#'] < TH_NEUT_SEVERE
    condition_mod_fever = (df_daily['minNEUT#'] < TH_NEUT_MODERATE) & (df_daily['maxBT'] >= TH_FEVER)
    df_daily['FN_Event'] = np.where(condition_severe | condition_mod_fever, 1, 0)

    # Filter records falling into the observation window (Day 1 to Day 28)
    df_base = df_chemo[['hs', 'DO_DATE']].drop_duplicates()
    df_merged = pd.merge(df_base, df_daily, on='hs', how='inner')
    
    # Calculate delta days relative to chemotherapy administration (DO_DATE)
    days_diff_raw = (df_merged['QUERY_DATE'] - df_merged['DO_DATE']).dt.days
    
    # Define Day 1 (chemotherapy day: diff = 0 -> Day 1) 
    df_merged['Day_Num'] = days_diff_raw + 1
    
    # Observation window criteria: Day 1 <= Day <= 28
    in_window = (df_merged['Day_Num'] >= 1) & (df_merged['Day_Num'] <= WINDOW_EVENT_END_DAY)
    df_merged['Valid_FN'] = np.where(in_window, df_merged['FN_Event'], 0)
    
    df_fn_res = df_merged.groupby(['hs', 'DO_DATE'])['Valid_FN'].sum().reset_index()

    # Aggregating therapeutic G-CSF events within Day 1 to Day 28
    df_gcsf_merged = pd.merge(df_base, df_gcsf_therapeutic, on='hs', how='left')
    
    gcsf_diff_raw = (df_gcsf_merged['GCSF_DATE'] - df_gcsf_merged['DO_DATE']).dt.days
    df_gcsf_merged['Day_Num'] = gcsf_diff_raw + 1
    
    # Event determination
    is_gcsf_event = (
        (df_gcsf_merged['Day_Num'] >= 1) & 
        (df_gcsf_merged['Day_Num'] <= WINDOW_EVENT_END_DAY) & 
        (df_gcsf_merged['EXEC_DOSE'] > 0)
    )
    df_gcsf_merged['Therapeutic_GCSF_Event'] = np.where(is_gcsf_event, 1, 0)
    df_gcsf_res = df_gcsf_merged.groupby(['hs', 'DO_DATE'])['Therapeutic_GCSF_Event'].sum().reset_index()

    # Consolidate data
    df_final = pd.merge(df_fn_res, df_gcsf_res, on=['hs', 'DO_DATE'], how='left')
    df_final['Event'] = ((df_final['Valid_FN'] + df_final['Therapeutic_GCSF_Event'].fillna(0)) >= 1).astype(int)
    df_final['Valid_FN'] = ((df_final['Valid_FN'].fillna(0)) >= 1).astype(int)
    df_final['Therapeutic_GCSF_Event'] = ((df_final['Therapeutic_GCSF_Event'].fillna(0)) >= 1).astype(int)
    
    return df_final[['hs', 'DO_DATE', 'Event','Valid_FN','Therapeutic_GCSF_Event']]

def create_features(df_base, dfs, df_chemo, df_glas_prophylactic):
    """
    Feature Engineering
    """
    # Extract the baseline (latest prior) laboratory values
    df_exam_last = get_latest_record_before_target(
        df_base, 
        dfs['検査値'][dfs['検査値']['EXAM_ITEM'].isin(REQ_EXAM_ITEMS)], 
        'QUERY_DATE'
    )
    df_feat_exam = (df_exam_last.groupby(['hs', 'DO_DATE', 'EXAM_ITEM'])['RESULT_NUM'].min()
                    .unstack().rename(columns={'  NEUT#': 'lastNEUT#', '  WBC': 'lastWBC'}).reset_index())
    df_feat_exam = pd.merge(df_feat_exam, df_chemo[['hs', 'DO_DATE', 'AGE']].drop_duplicates(), on=['hs', 'DO_DATE'])

    # Extract the baseline somatometry values (using forward-fill)
    df_soma = dfs['身体計測'].sort_values(['hs', 'SOMA_DATE']).copy()
    for col in ['MS_HEIGHT', 'MS_WEIGHT', 'MS_BSA']:
        df_soma[col] = df_soma.groupby('hs')[col].ffill()
    
    df_soma_last = get_latest_record_before_target(df_base, df_soma, 'SOMA_DATE')
    df_feat_soma = df_soma_last.groupby(['hs', 'DO_DATE'])[['MS_HEIGHT', 'MS_WEIGHT']].median().reset_index()

    # Extract the baseline vital signs
    df_vt_last = get_latest_record_before_target(df_base, dfs['バイタル'], 'QUERY_DATE')
    
    vital_cols = [
    "TEMPR",       # 体温
    "PULSE",       # 脈拍
    "RESP",        # 呼吸数
    "BPH",         # 血圧（High/収縮期）
    "BPL"          # 血圧（Low/拡張期）
    ]

    # Target only the columns actually present in the DataFrame
    target_cols = [col for col in vital_cols if col in df_vt_last.columns]

    # Group by patient ID (hs) and chemotherapy date (DO_DATE) to calculate the median
    df_vt_feature = df_vt_last.groupby(['hs', 'DO_DATE'])[target_cols].median().reset_index()

    # Cumulative chemotherapy dosage
    df_feat_drug = (df_chemo.groupby(['hs', 'DO_DATE', 'DRUG_CD'])['EXEC_DOSE'].sum()
                    .unstack(fill_value=0).reset_index())

    # Baseline prophylactic G-CSF dosage prior to Day 1
    df_glas_pre = pd.merge(df_base, df_glas_prophylactic, on='hs', how='left')
    
    days_diff_raw = (df_glas_pre['GLAS_DATE'] - df_glas_pre['DO_DATE']).dt.days
    df_glas_pre['Day_Num'] = days_diff_raw + 1
    
    # Define time-window for prophylactic G-CSF
    in_range = (
        (df_glas_pre['Day_Num'] >= (1 - WINDOW_PRE_GCSF_DAYS + 1)) & 
        (df_glas_pre['Day_Num'] < 1)
    )
    df_glas_pre.loc[~in_range, 'EXEC_DOSE'] = 0
    
    df_feat_glas = df_glas_pre.groupby(['hs', 'DO_DATE'])['EXEC_DOSE'].sum().reset_index(name='TOTAL_GLAS_PREDOSE')

    # Patient Sex (M: 0, F: 1)
    df_sex = dfs['検査値'][['hs', 'PT_SEX']].drop_duplicates()
    df_sex['PT_SEX'] = df_sex['PT_SEX'].map({'M': 0, 'F': 1})

    return {
        'exam': df_feat_exam, 'soma': df_feat_soma, 'vt': df_vt_feature,
        'drug': df_feat_drug, 'glas': df_feat_glas, 'sex': df_sex
    }

In [ ]:
# ==========================================
# 4. Main Execution Pipeline 
# ==========================================
def main():
    # Load integrated datasets
    dfs = load_integrated_data(INPUT_DIR, FILE_NAMES)
    if not dfs or dfs['がん登録'].empty:
        print("Error: Cancer data is missing.")
        return
    
    # Filter patients to isolate solid tumors 
    df_cancer_feat = filter_cancer_patients(dfs['がん登録'])

    # Preprocess drugs and isolate chemotherapy recipients
    df_chemo, df_gcsf_therapeutic, df_glas_prophylactic = process_drugs(dfs)
    df_chemo, df_gcsf, df_glas = process_drugs(dfs)

    # Construct target labels and baseline features 
    df_labels = create_event_labels(df_chemo, dfs['検査値'], dfs['バイタル'], df_gcsf_therapeutic)
    df_base = df_labels[['hs', 'DO_DATE']].drop_duplicates()
    features = create_features(df_base, dfs, df_chemo, df_glas_prophylactic)

    # Merge features and labels
    df_final = df_labels.copy()
    for name, df_feat in features.items():
        join_keys = ['hs', 'DO_DATE'] if 'DO_DATE' in df_feat.columns else ['hs']
        if not df_feat.empty:
            # Inner join ensures exclusion of samples with incomplete mandatory data
            df_final = pd.merge(df_final, df_feat, on=join_keys, how='inner') 
    if not df_cancer_feat.empty:
        df_final = pd.merge(df_final, df_cancer_feat, on='hs', how='inner')

    # Export processed cohort dataset
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    output_filename = 'step2_all_tagt_feat'
    df_final = df_final.drop(columns=['Valid_FN', 'Therapeutic_GCSF_Event'])
    df_final.to_parquet(f'{OUTPUT_DIR}/{output_filename}')
    print(f"Saved to: {OUTPUT_DIR}/{output_filename}") 

if __name__ == "__main__":
    main()